# Bradley-Terry Models: VFL 1916 Example

In [10]:
import os
import sys

root_dir = os.path.dirname(os.path.abspath(os.curdir))
src_dir = os.path.join(root_dir, 'src', 'python')
sys.path.append(src_dir)
# print(sys.path[-1])

This is a brief example using our own package to fit a simple BT model to the VFL 1916 data.

In [11]:
import pandas as pd
from utils import make_win_matrix, make_victory_totals

path = "../data/VFL1916.csv"
vfl_df = pd.read_csv(path, header=0)
vfl_mat = make_win_matrix(vfl_df)
vfl_mat

,Carlton,Collingwood,Fitzroy,Richmond
Carlton,0.0,3.0,3.0,4.0
Collingwood,1.0,0.0,3.5,2.0
Fitzroy,1.0,0.5,0.0,1.0
Richmond,0.0,2.0,3.0,0.0


In [12]:
make_victory_totals(vfl_mat)

,Team,Wins,Losses,Total,Win Rate
0,Carlton,10.0,2.0,12.0,0.833333
1,Collingwood,6.5,5.5,12.0,0.541667
2,Fitzroy,2.5,9.5,12.0,0.208333
3,Richmond,5.0,7.0,12.0,0.416667


## Simple Bradley-Terry Model

In [13]:
from StaticModel import VANBT

mod = VANBT()
mod.fit(vfl_mat)
mod._fit_summary

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 12.522717844134803
        x: [ 2.365e+00  1.229e+00  0.000e+00  7.876e-01]
      nit: 7
      jac: [ 3.492e-06 -6.453e-07 -1.292e-06 -1.555e-06]
 hess_inv: [[ 6.040e-01  1.652e-01  9.332e-02  1.375e-01]
            [ 1.652e-01  4.855e-01  1.609e-01  1.883e-01]
            [ 9.332e-02  1.609e-01  5.609e-01  1.848e-01]
            [ 1.375e-01  1.883e-01  1.848e-01  4.894e-01]]
     nfev: 10
     njev: 10

In [14]:
mod.get_ranking()

,Team,Ability
0,Carlton,2.365038
1,Collingwood,1.228527
3,Richmond,0.787608
2,Fitzroy,0.000000


## Home-ground Advantage (HGA)

Alternatively, home-field advantage (HFA). The data requirements are different for HFA models, since we need to block wins by the venue. Our model considers neutral venues, but we will assume there are none for the VFL data. Currently, we will just need to add a dummy column of zeroess to indicate which matches are neutral (by default assuming the first column is the home team).

In [15]:
from utils import make_venue_win_matrices

vfl_df["is_neutral"] = 0
vfl_venue_wins = make_venue_win_matrices(vfl_df)

for i in vfl_venue_wins.items():
    print(i[0])
    print(i[1])
    print()

home
             Carlton  Collingwood  Fitzroy  Richmond
Carlton          0.0          1.0      1.0       2.0
Collingwood      0.0          0.0      2.0       1.0
Fitzroy          0.0          0.5      0.0       1.0
Richmond         0.0          1.0      2.0       0.0

away
             Carlton  Collingwood  Fitzroy  Richmond
Carlton          0.0          2.0      2.0       2.0
Collingwood      1.0          0.0      1.5       1.0
Fitzroy          1.0          0.0      0.0       0.0
Richmond         0.0          1.0      1.0       0.0

neutral
             Carlton  Collingwood  Fitzroy  Richmond
Carlton          0.0          0.0      0.0       0.0
Collingwood      0.0          0.0      0.0       0.0
Fitzroy          0.0          0.0      0.0       0.0
Richmond         0.0          0.0      0.0       0.0



In [16]:
from StaticModel import CHABT

mod_hfa = CHABT()
mod_hfa.fit(vfl_venue_wins, verbose=True)

Optimization terminated successfully.
         Current function value: 12.492423
         Iterations: 9
         Function evaluations: 13
         Gradient evaluations: 13
Successfully fit. Summary below:

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 12.492422600796797
        x: [ 2.372e+00  1.232e+00  0.000e+00  7.901e-01 -1.213e-01]
      nit: 9
      jac: [ 1.146e-08  6.391e-08 -4.843e-08 -2.694e-08  4.457e-07]
 hess_inv: [[ 6.065e-01  1.642e-01 ...  1.377e-01 -1.554e-02]
            [ 1.642e-01  4.866e-01 ...  1.875e-01 -3.919e-03]
            ...
            [ 1.377e-01  1.875e-01 ...  4.903e-01  4.335e-03]
            [-1.554e-02 -3.919e-03 ...  4.335e-03  2.322e-01]]
     nfev: 13
     njev: 13


In [17]:
print(mod_hfa.get_ranking())
print()
print(mod_hfa.params[-1])

          Team   Ability
0      Carlton  2.372262
1  Collingwood  1.232391
3     Richmond  0.790082
2      Fitzroy  0.000000

-0.1213094947992217


In [18]:
from utils import check_model_grad
from StaticModel import CHABT

model = CHABT()
model._set_data(vfl_venue_wins)
check_model_grad(model)

np.float64(1.3583496858140792e-06)